# 07 — Analytical Dashboard Data & Financial Analysis

Computes:
1. Edge risk scores (source/target risk mapping)
2. Money flow per node
3. Loss vs savings financial analysis
4. Node risk profiles

**Pre-computed scores** loaded from notebook 06 output.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Pipeline integration ──────────────────────────────────────────
_RUN_DIR = os.environ.get("AML_RUN_DIR", "")
DATA_DIR = os.path.join(_RUN_DIR, "data") if _RUN_DIR else "data"
MODELS_DIR = os.path.join(_RUN_DIR, "models") if _RUN_DIR else "models"
RESULTS_DIR = os.path.join(_RUN_DIR, "results") if _RUN_DIR else "results"

os.makedirs(RESULTS_DIR, exist_ok=True)

DEMO_CONFIG = {
    'avg_loss_per_undetected_aml': 0.15,
    'investigation_cost_per_alert': 500,
    'false_positive_cost': 200,
    'regulatory_fine_multiplier': 3.0,
    'recovery_rate_detected': 0.70,
    'currency': 'USD'
}

print("Dashboard data pipeline initialized!")

In [ ]:
# Load all data (using pre-computed scores from notebook 06)
transactions = pd.read_parquet(os.path.join(DATA_DIR, "transactions_processed.parquet"))
node_features_df = pd.read_parquet(os.path.join(DATA_DIR, "node_features.parquet"))
node_embeddings = pd.read_parquet(os.path.join(RESULTS_DIR, "node_embeddings_scored.parquet"))

assert "anomaly_score" in node_embeddings.columns, "anomaly_score missing — run notebook 06 first"
assert "is_anomaly" in node_embeddings.columns, "is_anomaly missing — run notebook 06 first"
assert "risk_score" in node_embeddings.columns, "risk_score missing — run notebook 06 first"

# Load threshold
X_train = np.load(os.path.join(MODELS_DIR, "X_train.npy"))
import torch
from gan_anomaly import Generator, Encoder, anomaly_score as compute_anomaly_score
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
with open(os.path.join(MODELS_DIR, "training_meta.json")) as f:
    meta = json.load(f)
G_m = Generator(meta["latent_dim"], meta["input_dim"], meta["g_hidden"], meta["n_layers"], meta["activation"]).to(device)
E_m = Encoder(meta["input_dim"], meta["latent_dim"], meta["e_hidden"], meta["n_layers"], meta["activation"]).to(device)
G_m.load_state_dict(torch.load(os.path.join(MODELS_DIR, "generator.pt"), map_location=device, weights_only=True))
E_m.load_state_dict(torch.load(os.path.join(MODELS_DIR, "encoder.pt"), map_location=device, weights_only=True))
train_scores = compute_anomaly_score(torch.tensor(X_train, dtype=torch.float32).to(device), E_m, G_m).cpu().numpy()
threshold = np.percentile(train_scores, 99)

print(f"Transactions: {len(transactions):,}")
print(f"Nodes: {len(node_embeddings):,}")
print(f"Anomalies: {node_embeddings['is_anomaly'].sum():,}")
print(f"Threshold: {threshold:.6f}")

In [ ]:
# Prepare transaction-level risk data
node_risk_dict = node_embeddings.set_index("id")["risk_score"].to_dict()
node_anomaly_dict = node_embeddings.set_index("id")["is_anomaly"].to_dict()

edges_df = transactions.copy()
edges_df["source_risk"] = edges_df["source"].map(node_risk_dict).fillna(0)
edges_df["target_risk"] = edges_df["target"].map(node_risk_dict).fillna(0)
edges_df["edge_risk"] = edges_df[["source_risk", "target_risk"]].max(axis=1)
edges_df["source_anomaly"] = edges_df["source"].map(node_anomaly_dict).fillna(False)
edges_df["target_anomaly"] = edges_df["target"].map(node_anomaly_dict).fillna(False)
edges_df["is_suspicious"] = edges_df["source_anomaly"] | edges_df["target_anomaly"]

# Node type onto edges
node_type_dict = node_features_df.set_index("id")["type"].to_dict()
edges_df["source_type"] = edges_df["source"].map(node_type_dict).fillna(-1).astype(int)
edges_df["target_type"] = edges_df["target"].map(node_type_dict).fillna(-1).astype(int)

# Money flow per node
outgoing = edges_df.groupby("source").agg({"base_amt": "sum", "tran_id": "count"}).rename(
    columns={"base_amt": "outgoing_amt", "tran_id": "outgoing_count"})
incoming = edges_df.groupby("target").agg({"base_amt": "sum", "tran_id": "count"}).rename(
    columns={"base_amt": "incoming_amt", "tran_id": "incoming_count"})

node_money = node_embeddings[["id", "anomaly_score", "is_anomaly", "risk_score"]].copy()
if "is_sar" in node_embeddings.columns:
    node_money["is_sar"] = node_embeddings["is_sar"]
node_money = node_money.merge(outgoing, left_on="id", right_index=True, how="left")
node_money = node_money.merge(incoming, left_on="id", right_index=True, how="left")
node_money = node_money.fillna(0)
node_money["total_volume"] = node_money["outgoing_amt"] + node_money["incoming_amt"]
node_money["total_transactions"] = node_money["outgoing_count"] + node_money["incoming_count"]
node_money["net_flow"] = node_money["incoming_amt"] - node_money["outgoing_amt"]
node_money = node_money.merge(node_features_df[["id", "type"]], on="id", how="left")

# Save
edges_df.to_parquet(os.path.join(RESULTS_DIR, "edges_enriched.parquet"), index=False)
node_money.to_parquet(os.path.join(RESULTS_DIR, "node_money_flow.parquet"), index=False)
print(f"Saved edges_enriched ({len(edges_df)} rows) and node_money_flow ({len(node_money)} rows)")

## Executive Summary KPIs

In [4]:
suspicious_vol = edges_df[edges_df["is_suspicious"]]["base_amt"].sum()
normal_vol = edges_df[~edges_df["is_suspicious"]]["base_amt"].sum()

print("Executive Summary KPIs:")
print(f"  Total Transactions:   {len(edges_df):,}")
print(f"  Total Volume:         ${edges_df['base_amt'].sum():,.0f}")
print(f"  Anomalies Detected:   {node_embeddings['is_anomaly'].sum():,}")
print(f"  Detection Rate:       {100*node_embeddings['is_anomaly'].mean():.1f}%")
print(f"  Suspicious Volume:    ${suspicious_vol:,.0f}")
print(f"  Normal Volume:        ${normal_vol:,.0f}")

Executive Summary KPIs:
  Total Transactions:   430,744
  Total Volume:         $231,281,011
  Anomalies Detected:   89
  Detection Rate:       1.2%
  Suspicious Volume:    $220,812,415
  Normal Volume:        $10,468,596


## Loss vs Savings Analysis

In [ ]:
def calculate_loss_savings():
    suspicious_txn_value = edges_df[edges_df["is_suspicious"]]["base_amt"].sum()
    n_anomalies = node_embeddings["is_anomaly"].sum()
    n_normal = len(node_embeddings) - n_anomalies

    if "is_sar" in node_embeddings.columns:
        true_pos = int(((node_embeddings["is_sar"] == 1) & (node_embeddings["is_anomaly"])).sum())
        false_pos = int(((node_embeddings["is_sar"] == 0) & (node_embeddings["is_anomaly"])).sum())
        false_neg = int(((node_embeddings["is_sar"] == 1) & (~node_embeddings["is_anomaly"])).sum())
        true_neg = int(((node_embeddings["is_sar"] == 0) & (~node_embeddings["is_anomaly"])).sum())
    else:
        est_rate = 0.10
        true_pos = int(n_anomalies * est_rate)
        false_pos = n_anomalies - true_pos
        false_neg = int(n_normal * 0.01)
        true_neg = n_normal - false_neg

    cfg = DEMO_CONFIG
    avg_suspicious_txn = suspicious_txn_value / max(n_anomalies, 1)
    potential_loss_detected = true_pos * avg_suspicious_txn * cfg["avg_loss_per_undetected_aml"]
    recovered_amount = potential_loss_detected * cfg["recovery_rate_detected"]

    normal_txn_value = edges_df[~edges_df["is_suspicious"]]["base_amt"].sum()
    avg_normal_txn = normal_txn_value / max(n_normal, 1)
    potential_loss_undetected = false_neg * avg_normal_txn * cfg["avg_loss_per_undetected_aml"]
    regulatory_fine_risk = potential_loss_undetected * cfg["regulatory_fine_multiplier"]

    investigation_cost = n_anomalies * cfg["investigation_cost_per_alert"]
    false_positive_cost = false_pos * cfg["false_positive_cost"]
    total_operational_cost = investigation_cost + false_positive_cost
    net_savings = recovered_amount - total_operational_cost

    return {
        "true_positives": true_pos, "false_positives": false_pos,
        "false_negatives": false_neg, "true_negatives": true_neg,
        "suspicious_txn_value": suspicious_txn_value,
        "potential_loss_detected": potential_loss_detected,
        "recovered_amount": recovered_amount,
        "potential_loss_undetected": potential_loss_undetected,
        "regulatory_fine_risk": regulatory_fine_risk,
        "investigation_cost": investigation_cost,
        "false_positive_cost": false_positive_cost,
        "total_operational_cost": total_operational_cost,
        "net_savings": net_savings,
    }

ls = calculate_loss_savings()

precision = ls["true_positives"] / max(ls["true_positives"] + ls["false_positives"], 1)
recall = ls["true_positives"] / max(ls["true_positives"] + ls["false_negatives"], 1)
f1 = 2 * (precision * recall) / max(precision + recall, 0.0001)

print("Loss vs Savings Analysis:")
print(f"  TP: {ls['true_positives']:,}  FP: {ls['false_positives']:,}  FN: {ls['false_negatives']:,}  TN: {ls['true_negatives']:,}")
print(f"  Precision: {precision:.2%}  Recall: {recall:.2%}  F1: {f1:.2%}")
print(f"  Recovered:    ${ls['recovered_amount']:,.0f}")
print(f"  Op Cost:      ${ls['total_operational_cost']:,.0f}")
print(f"  Net Savings:  ${ls['net_savings']:,.0f}")

with open(os.path.join(RESULTS_DIR, "financial_metrics.json"), "w") as f:
    json.dump(ls, f, indent=2, default=str)
print(f"\nSaved {RESULTS_DIR}/financial_metrics.json")

## Risk Network & Node Profiles

In [6]:
high_risk_edges = edges_df[edges_df["edge_risk"] > 0.75]
suspicious_edges = edges_df[edges_df["is_suspicious"]]

print("Risk Network Summary:")
print(f"  Total edges:          {len(edges_df):,}")
print(f"  Suspicious edges:     {len(suspicious_edges):,} ({100*len(suspicious_edges)/len(edges_df):.1f}%)")
print(f"  High-risk edges:      {len(high_risk_edges):,} ({100*len(high_risk_edges)/len(edges_df):.1f}%)")

risk_categories = pd.cut(edges_df["edge_risk"], bins=[0, 0.25, 0.5, 0.75, 1.0],
                         labels=["Low", "Medium", "High", "Critical"])
risk_vol = edges_df.groupby(risk_categories)["base_amt"].agg(["sum", "mean", "count"])
print("\nTransaction Volume by Risk Category:")
print(risk_vol.to_string())

Risk Network Summary:
  Total edges:          430,744
  Suspicious edges:     412,519 (95.8%)
  High-risk edges:      231,672 (53.8%)

Transaction Volume by Risk Category:
                    sum        mean   count
edge_risk                                  
Low        4.653136e+07  550.035586   84597
Medium     2.657432e+07  533.878210   49776
High       3.453846e+07  533.832981   64699
Critical   1.236369e+08  533.672038  231672


In [7]:
high_risk = node_money[node_money["risk_score"] > 0.75]
med_risk = node_money[(node_money["risk_score"] > 0.25) & (node_money["risk_score"] <= 0.75)]
low_risk = node_money[node_money["risk_score"] <= 0.25]

print("Node Risk Profiles:")
print(f"  Low risk:    {len(low_risk):,} nodes, avg vol ${low_risk['total_volume'].mean():,.0f}")
print(f"  Medium risk: {len(med_risk):,} nodes, avg vol ${med_risk['total_volume'].mean():,.0f}")
print(f"  High risk:   {len(high_risk):,} nodes, avg vol ${high_risk['total_volume'].mean():,.0f}")

Node Risk Profiles:
  Low risk:    7,496 nodes, avg vol $28,096
  Medium risk: 3 nodes, avg vol $42,771,661
  High risk:   1 nodes, avg vol $123,636,868


In [8]:
print("\n" + "=" * 80)
print(" AML ANALYTICAL DASHBOARD - SUMMARY ")
print("=" * 80)
print(f"  Nodes: {len(node_embeddings):,}  |  Transactions: {len(edges_df):,}  |  Volume: ${edges_df['base_amt'].sum():,.0f}")
print(f"  Anomalies: {node_embeddings['is_anomaly'].sum():,}  |  Suspicious Vol: ${ls['suspicious_txn_value']:,.0f}")
print(f"  Net Savings: ${ls['net_savings']:,.0f}")
print("\nOutputs: edges_enriched.parquet, node_money_flow.parquet, financial_metrics.json")
print("=" * 80)


 AML ANALYTICAL DASHBOARD - SUMMARY 
  Nodes: 7,500  |  Transactions: 430,744  |  Volume: $231,281,011
  Anomalies: 89  |  Suspicious Vol: $220,812,415
  Net Savings: $4,630,463

Outputs: edges_enriched.parquet, node_money_flow.parquet, financial_metrics.json


In [9]:
# ── GPU Cleanup — free VRAM for next notebook ──
import gc
for v in ["G_m", "E_m"]:
    if v in dir():
        exec(f"del {v}")
torch.cuda.empty_cache(); gc.collect()
print(f"GPU freed: {torch.cuda.memory_allocated()/1e6:.1f} MB allocated")

GPU freed: 8.5 MB allocated
